# Gran Premi RL — entrena el teu cotxe

Aquesta és la plantilla per entrenar el teu agent per al concurs final del
bloc de Reinforcement Learning. El reglament complet (puntuació, format de
la cursa, terminis) és a `README.md`, a la mateixa carpeta. Llegeix-lo abans
de començar.

**Resum de les regles clau:**
- Entorn: `Racing` (`03_proyectos_dqn/racing/racing_game.py`), el mateix que ja
  coneixes del bloc 3.
- El teu agent és una xarxa `CarDQN` amb **6 entrades i 9 sortides**. No
  canviïs aquesta forma ni les mides de les capes amagades (128, 128): el
  jutge (`avalua_agent.py`) assumeix exactament aquesta arquitectura.
- Has d'entregar un fitxer anomenat **`agent.pth`** (l'`state_dict` de la
  xarxa, no l'agent sencer).
- Pots entrenar a un portàtil normal o a Google Colab (gratuït, sense GPU):
  aquest notebook funciona en tots dos, sempre que deixis `render=False`
  durant l'entrenament llarg (a Colab no hi ha pantalla).

In [ ]:
import sys, os
from pathlib import Path

AQUI = Path.cwd()
CARPETA_RL = AQUI.parent if AQUI.name == "06_concurs" else AQUI  # per si es mou el notebook
RACING_DIR = CARPETA_RL / "03_proyectos_dqn" / "racing"
CONCURS_DIR = CARPETA_RL / "06_concurs"

sys.path.insert(0, str(RACING_DIR))
sys.path.insert(0, str(CONCURS_DIR))

import racing_game as rg
print("racing_game importat correctament des de:", RACING_DIR)
print("Torch disponible:", rg.TORCH_AVAILABLE)

## L'entorn en dos minuts

- **Estat (6 valors)**: 5 sensors del cotxe (distància fins a la vora de la
  pista, normalitzada 0-1) + la velocitat actual normalitzada.
- **Accions (9)**: `0=res, 1=accelerar, 2=frenar, 3=esquerra, 4=dreta,
  5=accel+esq, 6=accel+dta, 7=fre+esq, 8=fre+dta`.
- **Recompensa**: +proporcional a la distància avançada cada pas, -10 si xoca.
- **Puntuació del concurs**: la `fitness` del cotxe al final de la partida,
  que en la pràctica és la distància recorreguda abans de xocar o
  d'esgotar el límit de passos (2000). Com més lluny arribis sense sortir-te
  de la pista, més puntues.

La classe `CarDQN` i l'agent `CarAgent` (DQN amb experience replay + target
network) ja existeixen a `racing_game.py`; els reutilitzem tal qual.

In [ ]:
agent = rg.CarAgent(agent_id=0, state_size=6, n_actions=9)

# Hiperparàmetres que pots retocar si vols experimentar (opcional):
# agent.gamma = 0.95            # factor de descompte
# agent.epsilon_decay = 0.995   # com de ràpid deixa d'explorar
# agent.epsilon_min = 0.05      # exploració mínima que mai abandona

print("Agent creat. Device:", agent.device)

In [ ]:
def entrena(agent, episodis, max_passos=2000, render=False, actualitza_cada=10, log_cada=10):
    """Bucle d'entrenament propi (no el de racing_game.py, que sempre obre
    finestra). Amb render=False no toca pygame per res: funciona igual a un
    portàtil que a Colab. Amb render=True veuràs el cotxe entrenant en viu
    (cal tenir pantalla; no funciona a Colab)."""
    joc = rg.RacingGame(n_cars=1, render=render)
    historial_fitness = []

    for ep in range(episodis):
        estats = joc.reset()
        passos = 0
        aturat = False

        while passos < max_passos and not aturat:
            if render:
                for event in rg.pygame.event.get():
                    if event.type == rg.pygame.QUIT:
                        aturat = True

            accio = agent.act(estats[0])
            seguents, recompenses, dones, all_done = joc.step([accio])

            agent.remember(estats[0], accio, recompenses[0], seguents[0], dones[0])
            agent.replay()

            estats = seguents
            passos += 1

            if render and passos % 3 == 0:
                joc.render()
                joc.clock.tick(120)

            if all_done:
                break

        agent.decay_epsilon()
        if ep % actualitza_cada == 0:
            agent.update_target()

        historial_fitness.append(joc.cars[0].fitness)

        if (ep + 1) % log_cada == 0:
            mitjana = sum(historial_fitness[-log_cada:]) / log_cada
            print(f"Episodi {ep + 1:4d} | fitness: {joc.cars[0].fitness:8.1f} | "
                  f"mitjana ({log_cada}): {mitjana:8.1f} | epsilon: {agent.epsilon:.3f}")

    joc.close()
    return historial_fitness

## Prova ràpida (no és l'entrenament de debò)

Aquesta primera crida és només per comprovar que tot funciona: pocs episodis
i pocs passos. Un cop vegis que corre sense errors, passa a la cel·la
següent i entrena de debò amb més episodis.

In [ ]:
historial = entrena(agent, episodis=5, max_passos=300, render=False)

## Entrenament de debò

Per un agent que arribi a puntuar bé sol caldre entre 300 i 600 episodis en
CPU (uns quants minuts). Puja el nombre d'episodis a mesura que tinguis
temps a classe; pots tornar a cridar `entrena(...)` diverses vegades sobre
el mateix `agent`, que continua aprenent des d'on ho havia deixat (no
reiniciïs la cel·la de dalt on es crea `agent` si no vols començar de zero).

Deixa `render=False` mentre entrenes de debò (molt més ràpid, i obligatori
a Colab). Activa `render=True` només si vols mirar-lo un moment.

In [ ]:
# historial += entrena(agent, episodis=300, max_passos=2000, render=False)

## Desar el model

Nom **fix**: `agent.pth`, a la mateixa carpeta que aquest notebook
(`06_concurs/`). Si li poses un altre nom o el guardes en un altre lloc,
el jutge no el trobarà.

In [ ]:
import torch

RUTA_SORTIDA = CONCURS_DIR / "agent.pth"
torch.save(agent.q_network.state_dict(), RUTA_SORTIDA)
print("Model desat a:", RUTA_SORTIDA)

## Autoavaluació abans d'entregar

Fes servir el mateix jutge que farà servir el professor per comprovar la
teva puntuació abans d'entregar. Si surt un error de càrrega, vol dir que
has tocat l'arquitectura de la xarxa (revisa que `CarAgent` no s'hagi
modificat) o que el fitxer no s'ha desat bé.

In [ ]:
import avalua_agent as jutge

resultat = jutge.avaluar_agent(str(RUTA_SORTIDA), nom="el meu agent")
jutge.imprimeix_resultat(resultat)

## Abans d'entregar, comprova...

- [ ] El fitxer es diu exactament `agent.pth`.
- [ ] `avalua_agent.py --model agent.pth` no dona error de càrrega.
- [ ] La fitness mitjana és clarament superior a la de l'"Agent aleatori"
      (si no, encara li falta entrenament: puja el nombre d'episodis).
- [ ] Has penjat `agent.pth` a la carpeta de lliuraments amb el teu nom
      abans de la sessió 4 (25 de gener), que és quan es tanca la
      classificació per a la Gran Cursa Final.